# Full-video pose (MMPose + MMDet)

Runs detection + pose for the target IDs using the beefier models from `cache_pose.py`.
- If a frame contains only the target ID(s), run detection once and pose on the full frame.
- Otherwise, use SAM3 boxes for the targets as pose bboxes (no detection).
- Outputs an H.264 MP4 with pose overlays.

In [1]:
from pathlib import Path
import subprocess
import h5py
import json

import cv2
import numpy as np
from tqdm import tqdm
from pycocotools import mask as maskUtils

from mmengine.registry import init_default_scope
from mmdet.apis import inference_detector, init_detector
from mmpose.apis import inference_topdown, init_model as init_pose_estimator

from sam3_crops_utils import (
    load_parsed_csv,
    load_rotation_report,
    get_video_paths,
    load_mask_cache,
    apply_rotation,
    rotate_boxes,
)

/home/brukew/miniconda3/envs/detpose/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/brukew/miniconda3/envs/detpose/lib/python3.10/site-packages/mmengine/utils/package_utils.py:48: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
# Model/config paths (from cache_pose.py)
DETECTION_CONFIG = "/orcd/data/satra/002/models/mmdet/dino-5scale_swin-l_8xb2-36e_coco.py"
DETECTION_CHECKPOINT = "/orcd/data/satra/002/models/mmdet/dino-5scale_swin-l_8xb2-36e_coco-5486e051.pth"
POSE_CONFIG = "/orcd/data/satra/002/models/mmpose/td-hm_hrnet-w48_dark-8xb32-210e_coco-wholebody-384x288.py"
POSE_CHECKPOINT = "/orcd/data/satra/002/models/mmpose/hrnet_w48_coco_wholebody_384x288_dark-f5726563_20200918.pth"
DEVICE = "cuda:0"

# Detection filtering
DET_CONF_THRESH = 0.5
DET_MIN_W = 50
DET_MIN_H = 50

# Pose filtering
POSE_CONF = 0.25

# Cache settings
CACHE_BASE_PATH = "/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking"

# Video metadata path
VIDEO_META_JSON = Path("/orcd/data/satra/001/users/brukew/actreg/dataprep/video_meta.json")

# Load shared data
videos_data = load_parsed_csv()
rotation_data = load_rotation_report()

# Load video metadata (contains rotation_meta, fps, etc.)
with open(VIDEO_META_JSON, 'r') as f:
    video_meta_data = json.load(f)
# Create lookup by row_idx
video_meta_lookup = {rec['row_idx']: rec for rec in video_meta_data['records']}

# Init models
detector = init_detector(DETECTION_CONFIG, DETECTION_CHECKPOINT, device=DEVICE)
pose_model = init_pose_estimator(POSE_CONFIG, POSE_CHECKPOINT, device=DEVICE)

Loads checkpoint by local backend from path: /orcd/data/satra/002/models/mmdet/dino-5scale_swin-l_8xb2-36e_coco-5486e051.pth
Loads checkpoint by local backend from path: /orcd/data/satra/002/models/mmpose/hrnet_w48_coco_wholebody_384x288_dark-f5726563_20200918.pth


/home/brukew/miniconda3/envs/detpose/lib/python3.10/site-packages/mmpose/datasets/datasets/utils.py:102: UserWarning: The metainfo config file "configs/_base_/datasets/coco_wholebody.py" does not exist. A matched config file "/home/brukew/miniconda3/envs/detpose/lib/python3.10/site-packages/mmpose/.mim/configs/_base_/datasets/coco_wholebody.py" will be used instead.
  warnings.warn(


In [3]:
def build_ffmpeg_writer(output_path: Path, width: int, height: int, fps: float, loglevel: str = "error"):
    cmd = [
        "ffmpeg", "-loglevel", loglevel, "-y",
        "-f", "rawvideo", "-vcodec", "rawvideo", "-pix_fmt", "bgr24",
        "-s", f"{width}x{height}", "-r", str(fps if fps > 0 else 30), "-i", "-",
        "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", str(output_path),
    ]
    return subprocess.Popen(cmd, stdin=subprocess.PIPE)


def expand_box(box, expand_ratio=0.5, frame_w=None, frame_h=None):
    """Expand a bounding box by a ratio, optionally clipping to frame bounds"""
    x1, y1, x2, y2 = box
    w = x2 - x1
    h = y2 - y1
    
    # Expand by ratio
    x1_new = x1 - w * expand_ratio / 2
    y1_new = y1 - h * expand_ratio / 2
    x2_new = x2 + w * expand_ratio / 2
    y2_new = y2 + h * expand_ratio / 2
    
    # Clip to frame bounds if provided
    if frame_w is not None and frame_h is not None:
        x1_new = max(0, x1_new)
        y1_new = max(0, y1_new)
        x2_new = min(frame_w, x2_new)
        y2_new = min(frame_h, y2_new)
    
    return np.array([x1_new, y1_new, x2_new, y2_new])


def draw_keypoints(frame_bgr, res, color, radius: int = 3, conf_thresh: float = 0.3):
    # Access keypoints from pred_instances (MMPose API)
    if not hasattr(res, 'pred_instances') or not hasattr(res.pred_instances, 'keypoints'):
        return

    keypoints_xy = res.pred_instances.keypoints[0]  # Shape: (num_keypoints, 2)
    keypoint_scores = res.pred_instances.keypoint_scores[0]  # Shape: (num_keypoints,)

    # Convert to numpy if needed
    if hasattr(keypoints_xy, 'cpu'):
        keypoints_xy = keypoints_xy.cpu().numpy()
    if hasattr(keypoint_scores, 'cpu'):
        keypoint_scores = keypoint_scores.cpu().numpy()

    # Draw keypoints
    for (x, y), score in zip(keypoints_xy, keypoint_scores):
        if score > conf_thresh:
            cv2.circle(frame_bgr, (int(x), int(y)), radius, color, -1, lineType=cv2.LINE_AA)


def mmpose_to_pose_dict(pose_res_list):
    """Convert MMPose results to dict format for caching"""
    pose_dicts = []
    for res in pose_res_list:
        bbox = res.pred_instances.bboxes[0]
        if hasattr(bbox, 'cpu'):
            bbox = bbox.cpu().numpy()
        
        keypoints_xy = res.pred_instances.keypoints[0]
        keypoint_scores = res.pred_instances.keypoint_scores[0]
        
        if hasattr(keypoints_xy, 'cpu'):
            keypoints_xy = keypoints_xy.cpu().numpy()
        if hasattr(keypoint_scores, 'cpu'):
            keypoint_scores = keypoint_scores.cpu().numpy()
        
        # Combine keypoints and scores: (N_kpts, 3) with [x, y, score]
        keypoints = np.concatenate([
            keypoints_xy,
            keypoint_scores.reshape(-1, 1)
        ], axis=1)
        
        pose_dicts.append({
            'keypoints': keypoints,
            'bbox': bbox,
            'metadata': {}
        })
    
    return pose_dicts


def get_sam3_pose_cache_path(video_basename: str, cache_base_path: str = CACHE_BASE_PATH) -> Path:
    """Generate cache path with SAM3 indicator"""
    cache_dir = Path(cache_base_path) / "pose_sam3" / video_basename
    det_name = Path(DETECTION_CONFIG).stem
    pose_name = Path(POSE_CONFIG).stem
    filename = f"{det_name}_{DET_CONF_THRESH}_{pose_name}_sam3guided.h5"
    return cache_dir / filename


def save_pose_cache(poses_dict: dict, cache_path: Path):
    """Save pose results to HDF5 cache"""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    
    with h5py.File(cache_path, 'w') as f:
        # Save metadata
        f.attrs['detection_config'] = DETECTION_CONFIG
        f.attrs['pose_config'] = POSE_CONFIG
        f.attrs['sam3_guided'] = True  # Mark as SAM3-guided
        
        for frame_idx, pose_list in poses_dict.items():
            frame_group = f.create_group(f'frame_{frame_idx}')
            
            for pose_idx, pose_result in enumerate(pose_list):
                pose_group = frame_group.create_group(f'pose_{pose_idx}')
                
                pose_group.create_dataset(
                    'keypoints', 
                    data=pose_result['keypoints'], 
                    compression='gzip', 
                    compression_opts=4
                )
                
                pose_group.create_dataset(
                    'bbox', 
                    data=pose_result['bbox'], 
                    compression='gzip', 
                    compression_opts=4
                )
    
    print(f"Saved SAM3-guided pose cache to: {cache_path}")


def load_pose_cache(cache_path: Path):
    """Load pose results from HDF5 cache"""
    if not cache_path.exists():
        return None
    
    poses = {}
    
    try:
        with h5py.File(cache_path, 'r') as f:
            for frame_key in f.keys():
                frame_idx = int(frame_key.split('_')[1])
                frame_group = f[frame_key]
                
                frame_poses = []
                for pose_key in sorted(frame_group.keys()):
                    pose_group = frame_group[pose_key]
                    
                    frame_poses.append({
                        'keypoints': pose_group['keypoints'][:],
                        'bbox': pose_group['bbox'][:]
                    })
                
                poses[frame_idx] = frame_poses
        
        return poses
        
    except Exception as e:
        print(f"Error loading pose cache: {e}")
        return None


def decode_masks_for_frame(cache, frame_idx: int, rotation: int, frame_w: int, frame_h: int, base_w: int, base_h: int):
    masks_out = []
    if cache is None or 'rles' not in cache or frame_idx not in cache['rles']:
        return masks_out
    rles = cache['rles'][frame_idx]
    for rle in rles:
        rle_dict = {"counts": rle.tobytes(), "size": [base_h, base_w]}
        m = maskUtils.decode(rle_dict)
        if m.ndim == 3:
            m = m[:, :, 0]
        if rotation:
            m = apply_rotation(m, rotation)
        if m.shape != (frame_h, frame_w):
            m = cv2.resize(m.astype(np.uint8), (frame_w, frame_h), interpolation=cv2.INTER_NEAREST)
        masks_out.append(m.astype(bool))
    return masks_out


def filter_detections(pred_instances):
    person_mask = pred_instances.labels == 0
    bboxes = pred_instances.bboxes[person_mask]
    scores = pred_instances.scores[person_mask]
    keep = []
    for i, (box, sc) in enumerate(zip(bboxes, scores)):
        if sc < DET_CONF_THRESH:
            continue
        w = box[2] - box[0]
        h = box[3] - box[1]
        if w < DET_MIN_W or h < DET_MIN_H:
            continue
        keep.append(i)
    return bboxes[keep], scores[keep]


def child_ids_for_time(intervals, t: float):
    ids = []
    for iv in intervals:
        start = iv.get('start_sec', 0)
        end = iv.get('end_sec') or float('inf')
        if start <= t < end:
            try:
                ids.append(int(iv['id']))
            except Exception:
                pass
    return ids


def run_pose_video_mmpose(
    row_idx: int,
    output_path: str | Path,
    target_ids=None,
    input_mode: str = "blur",  # blur | black | raw
    buffer_px: int = 30,
    blur_ksize: int = 61,
    falloff_px: float = 120.0,
    pad: int = 20,
    conf: float = POSE_CONF,
    device: str | None = DEVICE,
    max_frames: int | None = None,
    ffmpeg_loglevel: str = "error",
    enable_cache: bool = True,
    force_recompute: bool = False,
    box_expand_ratio: float = 0.5,  # Expand SAM3 boxes by 50%
):
    row = videos_data[row_idx - 1]
    intervals = row.get('intervals', [])

    original_path, sam3_path, cache_path = get_video_paths(row_idx, videos_data)
    if not original_path:
        print("Missing original video")
        return

    cache = load_mask_cache(cache_path, video_basename=sam3_path.stem if sam3_path else None, prompt="person", model_name="facebook-sam3")
    cache_width = cache_height = None
    if cache and cache.get('attrs'):
        cache_width = cache['attrs'].get('width')
        cache_height = cache['attrs'].get('height')
    base_w = cache_width or 0
    base_h = cache_height or 0

    # Setup pose caching
    video_basename = original_path.stem
    pose_cache_path = get_sam3_pose_cache_path(video_basename)
    poses_to_save = {}
    cached_poses = None
    
    if enable_cache and not force_recompute:
        cached_poses = load_pose_cache(pose_cache_path)
        if cached_poses:
            print(f"Loaded {len(cached_poses)} cached pose frames from {pose_cache_path}")

    cap = cv2.VideoCapture(str(original_path))
    if not cap.isOpened():
        print("Could not open video")
        return
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total = min(frame_count, max_frames) if max_frames else frame_count

    # Get rotation metadata from video_meta.json instead of ffprobe
    video_meta = video_meta_lookup.get(row_idx, {})
    rotation_meta = video_meta.get('rotation_meta', 0) * -1 if video_meta.get('rotation_meta') else 0

    out_path = Path(output_path)
    writer = None

    for idx in tqdm(range(total), desc=f"row{row_idx}", leave=False):
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame_w, frame_h = frame_rgb.shape[1], frame_rgb.shape[0]

        base_width = cache_width or frame_w
        base_height = cache_height or frame_h
        auto_rotated_guess = bool(rotation_meta) and frame_w == base_height and frame_h == base_width
        rotate_frame = bool(rotation_meta) and not auto_rotated_guess
        rotate_boxes_flag = bool(rotation_meta)

        if rotate_frame:
            frame_rgb = apply_rotation(frame_rgb, rotation_meta)
            frame_w, frame_h = frame_rgb.shape[1], frame_rgb.shape[0]

        # Get SAM3 cache boxes for this frame
        boxes = obj_ids = scores = None
        masks = None
        key = idx
        if cache and key in cache.get('boxes', {}):
            boxes = cache['boxes'][key]
            obj_ids = cache['obj_ids'][key]
            scores = cache['scores'][key]
            masks = decode_masks_for_frame(cache, key, rotation_meta if rotate_boxes_flag else 0, frame_w, frame_h, base_width, base_height)

        # Rotate/scale boxes
        boxes_rot = None
        if boxes is not None:
            boxes_rot = boxes.copy()
            expected_w, expected_h = base_width, base_height
            if rotate_boxes_flag and rotation_meta:
                boxes_rot = rotate_boxes(boxes_rot, rotation_meta, base_width, base_height)
                if rotation_meta in (-90, 90, -270, 270):
                    expected_w, expected_h = base_height, base_width
                elif rotation_meta in (-180, 180):
                    expected_w, expected_h = base_width, base_height
            if expected_w and expected_h and (frame_w != expected_w or frame_h != expected_h):
                scale_x = frame_w / expected_w
                scale_y = frame_h / expected_h
                boxes_rot[:, [0, 2]] *= scale_x
                boxes_rot[:, [1, 3]] *= scale_y
                if masks:
                    masks = [cv2.resize(m.astype(np.uint8), (frame_w, frame_h), interpolation=cv2.INTER_NEAREST).astype(bool) for m in masks]

        frame_time = idx / fps if fps else 0.0
        target_set = set(int(t) for t in (target_ids or child_ids_for_time(intervals, frame_time)) if t is not None)

        if boxes_rot is None or obj_ids is None:
            if writer is None:
                writer = build_ffmpeg_writer(out_path, frame_w, frame_h, fps, loglevel=ffmpeg_loglevel)
            writer.stdin.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR).tobytes())
            continue

        keep = [i for i, oid in enumerate(obj_ids) if int(oid) in target_set]
        if not keep:
            if writer is None:
                writer = build_ffmpeg_writer(out_path, frame_w, frame_h, fps, loglevel=ffmpeg_loglevel)
            writer.stdin.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR).tobytes())
            continue

        frame_vis = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        # Check if we have cached poses for this frame
        if cached_poses and idx in cached_poses:
            # Use cached poses
            for cached_pose in cached_poses[idx]:
                # Draw from cached keypoints
                keypoints = cached_pose['keypoints']
                for (x, y, score) in keypoints:
                    if score > conf:
                        cv2.circle(frame_vis, (int(x), int(y)), 3, (0, 255, 0), -1, lineType=cv2.LINE_AA)
        else:
            # Compute poses - build union box over all target boxes and expand
            xs1 = [boxes_rot[ki][0] for ki in keep]
            ys1 = [boxes_rot[ki][1] for ki in keep]
            xs2 = [boxes_rot[ki][2] for ki in keep]
            ys2 = [boxes_rot[ki][3] for ki in keep]
            union_box = [min(xs1), min(ys1), max(xs2), max(ys2)]
            
            # Expand the union box by the specified ratio
            expanded_box = expand_box(union_box, expand_ratio=box_expand_ratio, frame_w=frame_w, frame_h=frame_h)
            
            # Run pose estimation on expanded box
            pose_res = inference_topdown(pose_model, frame_vis, expanded_box.reshape(1, 4))
            
            # Draw and save to cache
            for res in pose_res:
                draw_keypoints(frame_vis, res, (0, 255, 0), conf_thresh=conf)
            
            # Save for caching
            if enable_cache:
                poses_to_save[idx] = mmpose_to_pose_dict(pose_res)

        if writer is None:
            writer = build_ffmpeg_writer(out_path, frame_w, frame_h, fps, loglevel=ffmpeg_loglevel)
        writer.stdin.write(frame_vis.astype(np.uint8).tobytes())

    if writer is not None:
        writer.stdin.close()
        writer.wait()
    cap.release()
    
    # Save pose cache
    if enable_cache and poses_to_save:
        save_pose_cache(poses_to_save, pose_cache_path)
    
    print(f"Saved pose video to {out_path}")

In [8]:
# Example:
run_pose_video_mmpose(row_idx=45, output_path="row45_pose_mmpose.mp4", target_ids=None, input_mode="blur", buffer_px=30, conf=0.55, force_recompute=True)

Saved SAM3-guided pose cache to: /orcd/scratch/bcs/001/sensein/sails/cache_for_tracking/pose_sam3/IMG_1161/dino-5scale_swin-l_8xb2-36e_coco_0.5_td-hm_hrnet-w48_dark-8xb32-210e_coco-wholebody-384x288_sam3guided.h5
Saved pose video to row45_pose_mmpose.mp4
